# 💬 Chatbot Starter

**AI Learning Playground — Educational Quickstart Blueprint**

Build, run, and deploy a conversational AI chatbot
using **Zephyr 7B Beta GGUF** running entirely on your GPU — no cloud API needed.

> ⚠️ **Prerequisite:** Run **[project-setup.ipynb](project-setup.ipynb)** first to install
> dependencies, validate your GPU, authenticate with Hugging Face, and download all models.

---

## What This Notebook Covers

| Step | Topic | Key Concept |
|------|-------|-------------|
| 1 | Configure Settings | Loading config from YAML, resolving model paths |
| 2 | Initialize Model | Instantiating `ChatbotModel` with `LlamaCpp` |
| 3 | Demo | Calling `model.predict()` with a DataFrame |
| 4 | GPU Monitoring | Tracking VRAM and power usage |
| 5 | Register Model | Logging to MLflow and registering as `AIStudio-EQ-Chatbot` |
| 6 | Verify | Loading the registered model and running a test inference |

## Architecture (v2.0.0)

```
  ChatbotModel (src/mlflow/models/chatbot.py)
       │
       ├── predict(DataFrame)     ← same method in notebook AND at serving time
       │
  Logger.log_model()             ← packages config + demo + code → MLflow artifact
       │
  mlflow.register_model()        ← creates AIStudio-EQ-Chatbot in Model Registry
       │
  mlflow models serve            ← AI Studio deploys this + demo/chatbot/main.py
```

**Key insight:** The `model.predict()` call in the Demo section is the EXACT same code
that runs when the model is deployed in production. Zero divergence.

In [ ]:
import sys
import time

# Ensure the project root is on the Python path so `src.*` imports work
sys.path.insert(0, "..")

start_time = time.time()
print("⏱️  Notebook started")

## 2. Configure Settings

Load the capability-specific configuration from `configs/chatbot.yaml`.
This YAML drives both the notebook demo AND the registered model — same settings, guaranteed.

In [ ]:
import os
from src.utils import load_config

config = load_config("../configs/chatbot.yaml")

# MODEL_ARTIFACTS_PATH is set by the MLflow serving container.
# During notebook use, it falls back to the model_path in config.yaml.
model_path     = os.environ.get("MODEL_ARTIFACTS_PATH", config.get("model_path", ""))
context_window = config.get("context_window", 8192)

print(f"Capability : {config.get('capability')}")
print(f"Model path : {model_path}")
print(f"Context    : {context_window} tokens")
print(f"UI mode    : {config.get('ui', {}).get('mode', 'streamlit')}")

## 3. Verify Assets

Confirm that the required model files are present in datafabric before loading.

In [ ]:
from src.utils import log_asset_status

assets = [
    {"name": "LLM (GGUF)",      "path": model_path,                       "required": True},
    {"name": "Config YAML",     "path": "../configs/chatbot.yaml",         "required": True},
    {"name": "Chatbot demo UI", "path": "../demo/chatbot/main.py",         "required": False},
]

log_asset_status(assets)

## 4. Initialize ChatbotModel

Instantiate the `ChatbotModel` class — the same class that will be registered in MLflow.

**Why use the Model class instead of LlamaCpp directly?**
- The model class encapsulates all inference logic in a single, testable unit
- The exact same `predict()` method runs in the notebook AND in production
- If you modify `ChatbotModel`, the registered model automatically benefits

In [ ]:
from src.mlflow.models.chatbot import ChatbotModel

print("Initializing ChatbotModel...")
print("(This loads the LLM weights — may take 30–60 seconds on first run)")

model = ChatbotModel(
    config=config,
    model_path=model_path,
)

print("\n✅ ChatbotModel ready")
print(f"   LLM loaded : {'yes' if model.llm is not None else 'no (check model_path)'}")

## 5. Demo: Conversational Q&A

Call `model.predict()` with a pandas DataFrame — exactly how MLflow calls it at serving time.

The ChatbotModel accepts two columns:
- `question`      — the user's message
- `system_prompt` — the LLM's persona (optional, has a sensible default)

In [ ]:
import pandas as pd

# Single question — the simplest form of a model.predict() call
result = model.predict(pd.DataFrame([{
    "question":      "What is the difference between supervised and unsupervised learning?",
    "system_prompt": "You are a helpful AI tutor. Explain concepts clearly with examples.",
}]))

print("Question: What is the difference between supervised and unsupervised learning?")
print("\n" + "─" * 60)
print(result["answer"].iloc[0])

In [ ]:
# Batch inference: multiple questions in a single predict() call
questions = [
    "What is a neural network?",
    "How does backpropagation work?",
    "What are activation functions and why are they important?",
]

batch_df = pd.DataFrame([{"question": q} for q in questions])
batch_results = model.predict(batch_df)

for i, (q, row) in enumerate(zip(questions, batch_results.itertuples()), 1):
    print(f"Q{i}: {q}")
    print(f"A{i}: {row.answer[:180]}...")
    print()

In [ ]:
import time
import plotly.graph_objects as go

# Measure response time for several queries to visualize LLM throughput
bench_questions = [
    "What is AI?",
    "Explain deep learning in one sentence.",
    "What is the difference between AI and ML?",
]

times   = []
lengths = []

for q in bench_questions:
    t0 = time.time()
    res = model.predict(pd.DataFrame([{"question": q}]))
    elapsed = time.time() - t0
    times.append(elapsed)
    lengths.append(len(res["answer"].iloc[0]))

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Response time (s)",
    x=[f"Q{i+1}" for i in range(len(bench_questions))],
    y=times,
    marker_color="#0096d6",
    text=[f"{t:.1f}s" for t in times],
    textposition="auto",
))
fig.add_trace(go.Bar(
    name="Response length (chars)",
    x=[f"Q{i+1}" for i in range(len(bench_questions))],
    y=lengths,
    marker_color="#00c2e0",
    yaxis="y2",
))
fig.update_layout(
    title="LLM Performance: Response Time vs. Length",
    xaxis_title="Query",
    yaxis=dict(title="Time (seconds)", color="#0096d6"),
    yaxis2=dict(title="Response length (chars)", overlaying="y", side="right", color="#00c2e0"),
    barmode="group",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

## 6. GPU Monitoring

Monitor GPU utilization, VRAM usage, and power consumption during inference.
This helps you understand how LlamaCpp uses your GPU hardware.

In [ ]:
from src.gpu_monitor import GPUMonitor

monitor = GPUMonitor()
monitor.display_dashboard()

## 7. Register with MLflow

Register this model in the MLflow Model Registry as **`AIStudio-EQ-Chatbot`**.

What gets registered:
- `ChatbotModel` source code (via `code_paths=["../src"]`)
- `configs/chatbot.yaml` (capability config)
- `demo/chatbot/` (Streamlit UI — AI Studio serves this after deployment)
- `loader.py` entry point (MLflow calls `_load_pyfunc()` to reconstruct the model)

After registration, AI Studio users can select `AIStudio-EQ-Chatbot` from the Model Registry
and click **Deploy** — the Streamlit app starts automatically.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

ARTIFACT_PATH = "AIStudio-EQ-Chatbot"   # Name of this model run artifact
MODEL_NAME    = "AIStudio-EQ-Chatbot"   # Registered model name in AI Studio

print(f"Artifact path  : {ARTIFACT_PATH}")
print(f"Registered as  : {MODEL_NAME}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")

### 7.1 Define Model Signature

A **ModelSignature** tells MLflow the exact input and output column names and types.
It validates incoming requests at serving time and documents the API contract.

For `ChatbotModel` we define only the columns it actually uses — tight schema = better validation.

In [ ]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# Input: only the columns ChatbotModel.predict() actually reads
input_schema = Schema([
    ColSpec("string", "question"),       # User's message
    ColSpec("string", "system_prompt"),  # LLM persona (optional — has default)
])

# Output: universal across all 4 model types for consistency
output_schema = Schema([
    ColSpec("string", "answer"),    # LLM response text
    ColSpec("string", "messages"),  # JSON-serialized conversation history
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print("Input schema:")
for col in input_schema.inputs:
    print(f"  {col.name:15} {col.type}")

print("\nOutput schema:")
for col in output_schema.inputs:
    print(f"  {col.name:15} {col.type}")

### 7.2 Log and Register

`Logger.log_model()` packages everything into a single MLflow artifact:
- Config YAML with `capability: chatbot` → `loader.py` will select `ChatbotModel`
- `demo/chatbot/` → AI Studio serves this as the Streamlit UI after deployment
- All source code under `../src` → bundled for reproducibility

`mlflow.register_model()` creates a versioned entry in the Model Registry.

In [ ]:
from src.mlflow.logger import Logger

with mlflow.start_run(run_name=f"register-{ARTIFACT_PATH}") as run:
    Logger.log_model(
        signature     = signature,
        artifact_path = ARTIFACT_PATH,
        config_path   = "../configs/chatbot.yaml",
        docs_path     = "../docs",
        model_path    = model_path,
        demo_folder   = "../demo/chatbot",
    )
    run_id = run.info.run_id

print(f"✅ Model logged | Run ID: {run_id}")

model_uri = f"runs:/{run_id}/{ARTIFACT_PATH}"
reg = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"✅ Registered  : {MODEL_NAME} v{reg.version}")
print(f"   Status      : {reg.status}")
print(f"   Model URI   : {model_uri}")

## 8. Verify Registration

Load the model back from MLflow and run a test prediction to confirm the full pipeline works.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

test_result = loaded_model.predict(pd.DataFrame([{
    "question":      "Quick check: What is 2 + 2? Answer in one word.",
    "system_prompt": "You are a concise assistant.",
}]))

print("✅ Loaded model response:")
print(test_result["answer"].iloc[0])

In [ ]:
elapsed = time.time() - start_time
print(f"⏱️  Total notebook time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

---

## ✅ What We Accomplished

| Step | Result |
|------|--------|
| Environment | CUDA verified, dependencies installed |
| ChatbotModel | Initialized with LlamaCpp (Blackwell-optimized) |
| Demo | Single Q&A + batch inference + performance chart |
| GPU Monitor | VRAM and power usage visible |
| Registration | `AIStudio-EQ-Chatbot` registered in Model Registry |
| Verification | Loaded model responded to test query |

## Next Steps

- **Explore another capability:** Open `image-gen-starter.ipynb`
- **Deploy in AI Studio:** Select `AIStudio-EQ-Chatbot` from the Model Registry → click Deploy
- **Customize the model:** Edit `src/mlflow/models/chatbot.py` and re-run from cell 10
- **Change the LLM:** Update `model_path` in `configs/chatbot.yaml` to use a different GGUF file